# 🚀 Image Captioning — GPU Model Trainer (Google Colab)

> **Before running:** Go to `Runtime → Change runtime type → T4 GPU`

This notebook will:
1. Verify GPU availability
2. Clone your project from GitHub
3. Install dependencies
4. Mount Google Drive to persist trained model
5. Run the full training pipeline with EarlyStopping & checkpoints
6. Plot training history

## ✅ Step 1 — Verify GPU

In [1]:
!nvidia-smi

Thu Jul  9 15:09:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print(f'Num GPUs Available: {len(gpus)}')
for gpu in gpus:
    print(f'  → {gpu}')

if not gpus:
    raise RuntimeError('❌ No GPU detected! Go to Runtime → Change runtime type → GPU')

for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

print('\n✅ GPU ready!')

## 📁 Step 2 — Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_SAVE_DIR = '/content/drive/MyDrive/imcaption_full'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f'📂 Model will be saved to: {DRIVE_SAVE_DIR}')

## 📦 Step 3 — Clone Repo & Install Dependencies

> 💡 Replace the `REPO_URL` with your actual GitHub repository URL.

In [ ]:
import os

REPO_URL = 'https://github.com/divyanshrajput118/imcaption.git' 

if not os.path.exists('/content/imcaption'):
    os.system(f'git clone {REPO_URL} /content/imcaption')
else:
    print('Repo already cloned. Pulling latest...')
    os.system('cd /content/imcaption && git pull')

os.chdir('/content/imcaption')
print(f'Working directory: {os.getcwd()}')

In [ ]:
!git pull

In [ ]:
!pip install -e . -q
!pip install -r requirements_colab.txt -q
print('✅ Dependencies installed!')

## 📥 Step 4 — Check / Load Artifacts

> ⚡ If you have artifacts saved in Drive from a previous run, set `COPY_FROM_DRIVE = True`.

In [ ]:
!python main.py

In [ ]:
import shutil, os

# Destination folder on Drive
dest_dir = '/content/drive/MyDrive/imcaption_all_files'
os.makedirs(dest_dir, exist_ok=True)

# Map of source artifact -> destination filename
files_to_save = {
    '/content/imcaption/artifacts/training/model.keras': 'model.keras',
    '/content/imcaption/artifacts/training/best_model.keras': 'best_model.keras',
    '/content/imcaption/artifacts/model_evaluation/scores.json': 'scores.json',
}

for src, fname in files_to_save.items():
    if os.path.exists(src):
        shutil.copy(src, os.path.join(dest_dir, fname))
        print(f"✅ Saved {fname} to Drive!")
    else:
        print(f"⚠️ Skipped (not found): {src}")

In [ ]:
import os

required = [
    'artifacts/data_ingestion/Images',
    'artifacts/data_transformation/vectorizer_data.pkl',
    'artifacts/data_transformation/train_images_captions.pkl',
    'artifacts/prepare_base_model/feature_extractor.keras',
    'artifacts/prepare_base_model/main_model.keras',
]

missing = [p for p in required if not os.path.exists(p)]

if missing:
    print('⚠️  Missing artifacts:')
    for m in missing:
        print(f'  ✗ {m}')
    print('\nEither set COPY_FROM_DRIVE=True below or run the full pipeline.')
else:
    print('✅ All required artifacts found!')